# Task 180: pysyd_astero — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Asteroseismology power spectrum analysis using pySYD

**Paper**: pySYD: Asteroseismology
**Repository**: https://github.com/ashleychontos/pySYD

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 14.60 dB (power spectrum)
- **SSIM**: N/A (1D spectrum)
- **Evaluation**: Asteroseismic parameter extraction (νmax RE, Δν RE, power spectrum CC)

### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 180: pysyd_astero — Asteroseismic parameter extraction
Inverse Problem: Fitting global oscillation parameters (numax, delta_nu)
from stellar power spectra.

Forward: Given (numax, delta_nu, amplitudes, Harvey params) → power spectrum
Inverse: Given power spectrum → estimate numax and delta_nu
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`harvey_comp`, `bg_model`, `gt_background`, `osc_modes`, `gauss_env`, `compute_cc`, `make_figure`, `main`

In [ ]:
def harvey_comp(freq, zeta, nc):
    """Single Harvey component: P(ν) = ζ / (1 + (ν/ν_c)²)"""
    return zeta / (1.0 + (freq / nc) ** 2)

def bg_model(freq, z1, nc1, z2, nc2, w):
    """Background = 2 Harvey components + white noise."""
    return harvey_comp(freq, z1, nc1) + harvey_comp(freq, z2, nc2) + w

def gt_background(freq):
    return bg_model(freq, HARVEY_ZETA1, HARVEY_NC1, HARVEY_ZETA2, HARVEY_NC2, WHITE_NOISE)

def osc_modes(freq, numax, dnu, sigma_env, height, width):
    """Lorentzian modes modulated by Gaussian envelope."""
    eps = 1.5
    modes = np.zeros_like(freq)
    n_lo = int(np.floor((numax - 4 * sigma_env) / dnu))
    n_hi = int(np.ceil((numax + 4 * sigma_env) / dnu))

    for n in range(max(1, n_lo), n_hi + 1):
        for ell, vis in [(0, 1.0), (1, 0.7), (2, 0.5)]:
            d02 = -0.15 * dnu if ell == 2 else 0.0
            nu_m = dnu * (n + ell / 2.0 + eps) + d02
            if nu_m < freq[0] or nu_m > freq[-1]:
                continue
            env = np.exp(-0.5 * ((nu_m - numax) / sigma_env) ** 2)
            modes += height * env * vis * width ** 2 / ((freq - nu_m) ** 2 + width ** 2)
    return modes

def gauss_env(freq, amp, center, sigma):
    return amp * np.exp(-0.5 * ((freq - center) / sigma) ** 2)

def compute_cc(a, b):
    a, b = a - np.mean(a), b - np.mean(b)
    d = np.sqrt(np.sum(a**2) * np.sum(b**2))
    return float(np.sum(a * b) / d) if d > 1e-30 else 0.0

def make_figure(freq, ps_true, ps_obs, numax_est, dnu_est,
                bg_fit, snr, lag, acf, metrics, path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"Task 180: Asteroseismic Parameter Extraction\n"
        f"νmax = {numax_est:.1f} μHz (GT: {GT_NUMAX}), "
        f"Δν = {dnu_est:.2f} μHz (GT: {GT_DELTA_NU})",
        fontsize=13, fontweight='bold')

    bg_true = gt_background(freq)
    om = (freq > 60) & (freq < 200)
    df = freq[1] - freq[0]

    ax = axes[0, 0]
    ax.loglog(freq, ps_obs, color='lightgray', lw=0.3, alpha=0.5,
              label='Observed', rasterized=True)
    ax.loglog(freq, ps_true, 'b-', lw=0.8, alpha=0.7, label='True spectrum')
    ax.loglog(freq, bg_true, 'g--', lw=1.5, label='True BG')
    ax.loglog(freq, bg_fit, 'r-', lw=1.5, label='Fitted BG')
    ax.set_xlabel('Frequency (μHz)')
    ax.set_ylabel('Power (ppm²/μHz)')
    ax.set_title('(a) Power Spectrum & Background')
    ax.legend(fontsize=8)
    ax.set_xlim([FREQ_MIN, FREQ_MAX])

    ax = axes[0, 1]
    exc = np.clip(ps_obs / bg_fit - 1.0, -1, 50)
    exc_sm = gaussian_filter1d(exc, int(2.0 / df))
    ax.plot(freq[om], exc[om], color='lightgray', lw=0.3, alpha=0.5, rasterized=True)
    ax.plot(freq[om], exc_sm[om], 'b-', lw=1.0, label='Smoothed')
    ax.axvline(numax_est, color='r', ls='--', lw=1.5, label=f'νmax={numax_est:.1f}')
    ax.axvline(GT_NUMAX, color='g', ls=':', lw=1.5, label=f'GT={GT_NUMAX}')
    ax.set_xlabel('Frequency (μHz)')
    ax.set_ylabel('SNR')
    ax.set_title('(b) Background-Subtracted')
    ax.legend(fontsize=8)

    ax = axes[1, 0]
    et = gauss_env(freq[om], 1, GT_NUMAX, SIGMA_ENV)
    ee = gauss_env(freq[om], 1, numax_est, SIGMA_ENV)
    ax.plot(freq[om], et, 'g-', lw=2, label='GT')
    ax.plot(freq[om], ee, 'r--', lw=2, label='Est')
    ax.fill_between(freq[om], et, alpha=0.2, color='green')
    ax.fill_between(freq[om], ee, alpha=0.2, color='red')
    ax.set_xlabel('Frequency (μHz)')
    ax.set_ylabel('Normalized Envelope')
    ax.set_title('(c) Envelope: GT vs Estimated')
    ax.legend(fontsize=8)
    ax.text(0.05, 0.9, f'νmax RE={metrics["numax_relative_error"]*100:.2f}%',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax = axes[1, 1]
    show = lag < 30
    sw = max(3, int(0.5 / df))
    asm = gaussian_filter1d(acf[show], sigma=sw)
    ax.plot(lag[show], acf[show], color='lightblue', lw=0.5)
    ax.plot(lag[show], asm, 'b-', lw=1.5, label='Smoothed ACF')
    ax.axvline(dnu_est, color='r', ls='--', lw=2, label=f'Δν={dnu_est:.2f}')
    ax.axvline(GT_DELTA_NU, color='g', ls=':', lw=2, label=f'GT={GT_DELTA_NU}')
    ax.set_xlabel('Lag (μHz)')
    ax.set_ylabel('Autocorrelation')
    ax.set_title('(d) ACF — Δν')
    ax.legend(fontsize=8)
    ax.text(0.05, 0.9, f'Δν RE={metrics["delta_nu_relative_error"]*100:.2f}%',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[INFO] Figure saved to {path}")

def main():
    os.makedirs('results', exist_ok=True)
    print("=" * 60)
    print("Task 180: pysyd_astero")
    print("=" * 60)

    freq, ps_true, ps_obs = synthesize_data()
    bg = gt_background(freq)
    ix = np.argmin(np.abs(freq - GT_NUMAX))
    print(f"BG@νmax={bg[ix]:.1f}, True@νmax={ps_true[ix]:.1f}, "
          f"Mode/BG≈{ps_true[ix]/bg[ix]-1:.2f}")

    numax_est, dnu_est, plin, bg_fit, snr, lag, acf = inverse_solve(freq, ps_obs)
    print(f"νmax_est={numax_est:.3f}, Δν_est={dnu_est:.3f}")

    metrics = evaluate(freq, numax_est, dnu_est, bg_fit)
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    with open('results/metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    make_figure(freq, ps_true, ps_obs, numax_est, dnu_est,
                bg_fit, snr, lag, acf, metrics, 'results/reconstruction_result.png')

    recon = bg_fit + osc_modes(freq, numax_est, dnu_est, SIGMA_ENV,
                                MODE_HEIGHT, MODE_WIDTH)
    np.save('results/ground_truth.npy', ps_true)
    np.save('results/reconstruction.npy', recon)

    print("\n" + "=" * 60)
    npass = 'PASS' if metrics['numax_RE_pass'] else 'FAIL'
    dpass = 'PASS' if metrics['delta_nu_RE_pass'] else 'FAIL'
    print(f"νmax RE: {metrics['numax_relative_error']*100:.2f}% {npass}")
    print(f"Δν RE:   {metrics['delta_nu_relative_error']*100:.2f}% {dpass}")
    print(f"BG PSNR: {metrics['background_PSNR_dB']:.2f} dB")
    print(f"Env CC:  {metrics['envelope_CC']:.4f}")
    print("=" * 60)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `forward_model`, `synthesize_data`

In [ ]:
def forward_model(freq):
    return gt_background(freq) + osc_modes(freq, GT_NUMAX, GT_DELTA_NU,
                                            SIGMA_ENV, MODE_HEIGHT, MODE_WIDTH)

def synthesize_data():
    rng = np.random.RandomState(SEED)
    freq = np.arange(FREQ_MIN, FREQ_MAX, FREQ_RES)
    ps_true = forward_model(freq)
    ps_obs = ps_true * rng.exponential(1.0, size=len(freq))
    return freq, ps_true, ps_obs

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

Functions: `inverse_solve`

In [ ]:
def inverse_solve(freq, ps_obs):
    df = freq[1] - freq[0]

    # ── smooth in log-space ──
    log_ps = np.log10(np.maximum(ps_obs, 1e-10))
    ps_heavy = 10 ** gaussian_filter1d(log_ps, int(30.0 / df))   # ~30 μHz
    ps_med = 10 ** gaussian_filter1d(log_ps, int(3.0 / df))      # ~3 μHz

    # ── fit background (log-space), exclude central band initially ──
    def log_bg(f, lz1, lnc1, lz2, lnc2, lw):
        return np.log10(bg_model(f, 10**lz1, 10**lnc1, 10**lz2, 10**lnc2, 10**lw))

    p0 = [np.log10(5000), np.log10(30), np.log10(2000), np.log10(120), np.log10(0.5)]
    blo = [1, 0, 1, 0.5, -3]
    bhi = [6, 3, 6, 3, 3]

    mask0 = (freq < 50) | (freq > 250)
    try:
        popt, _ = curve_fit(log_bg, freq[mask0],
                            np.log10(np.maximum(ps_heavy[mask0], 1e-10)),
                            p0=p0, bounds=(blo, bhi), maxfev=30000)
    except Exception:
        popt = p0

    plin = [10**p for p in popt]
    bg_fit = bg_model(freq, *plin)

    # ── iterative: exclude oscillation bump, refit ──
    snr_r = ps_heavy / bg_fit
    bump = snr_r > 1.15
    bump_exp = binary_dilation(bump, structure=np.ones(int(15.0 / df)))
    mask1 = ~bump_exp
    if np.sum(mask1) > 200:
        try:
            popt2, _ = curve_fit(log_bg, freq[mask1],
                                 np.log10(np.maximum(ps_heavy[mask1], 1e-10)),
                                 p0=popt, bounds=(blo, bhi), maxfev=30000)
            plin = [10**p for p in popt2]
            bg_fit = bg_model(freq, *plin)
            popt = popt2
        except Exception:
            pass

    # ── SNR ──
    snr = ps_med / bg_fit - 1.0
    snr = np.maximum(snr, 0.0)

    # ── fit Gaussian → numax ──
    sel = (freq > 50) & (freq < 250)
    fs, ss = freq[sel], snr[sel]
    try:
        ip = np.argmax(ss)
        penv, _ = curve_fit(gauss_env, fs, ss,
                            p0=[ss[ip], fs[ip], 20.0],
                            bounds=([0, 50, 3], [1e6, 250, 80]), maxfev=30000)
        numax_est = penv[1]
        sig_est = abs(penv[2])
    except Exception:
        numax_est = fs[np.argmax(ss)]
        sig_est = 20.0

    # ── ACF → delta_nu ──
    half = max(2.5 * sig_est, 35.0)
    band = (freq > numax_est - half) & (freq < numax_est + half)
    excess = ps_obs[band] / bg_fit[band] - 1.0
    n = len(excess)
    ec = excess - np.mean(excess)
    fft_e = np.fft.rfft(ec, n=2 * n)
    acf_full = np.fft.irfft(np.abs(fft_e) ** 2)[:n]
    acf = acf_full / (acf_full[0] + 1e-30)
    lag = np.arange(n) * df

    lo, hi = 5.0, 20.0
    lm = (lag >= lo) & (lag <= hi)
    if np.any(lm):
        ls, acs = lag[lm], acf[lm]
        sw = max(3, int(0.5 / df))
        asm = gaussian_filter1d(acs, sigma=sw)
        pks, _ = find_peaks(asm, height=0.005, distance=int(2.0 / df))
        if len(pks) > 0:
            best = pks[np.argmax(asm[pks])]
            dnu_est = ls[best]
        else:
            dnu_est = ls[np.argmax(asm)]
    else:
        dnu_est = 10.0

    # parabolic refinement
    pi_ = np.argmin(np.abs(lag - dnu_est))
    if 1 < pi_ < len(acf) - 1:
        y0, y1, y2 = acf[pi_ - 1], acf[pi_], acf[pi_ + 1]
        d = 2.0 * (2.0 * y1 - y0 - y2)
        if abs(d) > 1e-10:
            dnu_est = lag[pi_] + (y0 - y2) / d * df

    return numax_est, dnu_est, plin, bg_fit, snr, lag, acf

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `evaluate`

In [ ]:
def compute_psnr(sig, ref):
    ls = np.log10(np.maximum(sig, 1e-10))
    lr = np.log10(np.maximum(ref, 1e-10))
    mse = np.mean((ls - lr) ** 2)
    if mse < 1e-30:
        return 100.0
    dr = np.max(lr) - np.min(lr)
    return 10.0 * np.log10(dr ** 2 / mse)

def evaluate(freq, numax_est, dnu_est, bg_fit):
    nre = abs(numax_est - GT_NUMAX) / GT_NUMAX
    dre = abs(dnu_est - GT_DELTA_NU) / GT_DELTA_NU
    psnr = compute_psnr(bg_fit, gt_background(freq))
    m = (freq > 60) & (freq < 200)
    cc = compute_cc(gauss_env(freq[m], 1, numax_est, SIGMA_ENV),
                    gauss_env(freq[m], 1, GT_NUMAX, SIGMA_ENV))
    return {
        "numax_true": GT_NUMAX,
        "numax_estimated": float(round(numax_est, 3)),
        "numax_relative_error": float(round(nre, 6)),
        "delta_nu_true": GT_DELTA_NU,
        "delta_nu_estimated": float(round(dnu_est, 3)),
        "delta_nu_relative_error": float(round(dre, 6)),
        "background_PSNR_dB": float(round(psnr, 2)),
        "envelope_CC": float(round(cc, 4)),
        "numax_RE_pass": bool(nre < 0.05),
        "delta_nu_RE_pass": bool(dre < 0.05),
    }

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
os.makedirs('results', exist_ok=True)
print("=" * 60)
print("Task 180: pysyd_astero")
print("=" * 60)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
freq, ps_true, ps_obs = synthesize_data()
bg = gt_background(freq)
ix = np.argmin(np.abs(freq - GT_NUMAX))
print(f"BG@νmax={bg[ix]:.1f}, True@νmax={ps_true[ix]:.1f}, "
      f"Mode/BG≈{ps_true[ix]/bg[ix]-1:.2f}")

numax_est, dnu_est, plin, bg_fit, snr, lag, acf = inverse_solve(freq, ps_obs)
print(f"νmax_est={numax_est:.3f}, Δν_est={dnu_est:.3f}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
metrics = evaluate(freq, numax_est, dnu_est, bg_fit)
for k, v in metrics.items():
    print(f"  {k}: {v}")

with open('results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

make_figure(freq, ps_true, ps_obs, numax_est, dnu_est,
            bg_fit, snr, lag, acf, metrics, 'results/reconstruction_result.png')

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
recon = bg_fit + osc_modes(freq, numax_est, dnu_est, SIGMA_ENV,
                            MODE_HEIGHT, MODE_WIDTH)
np.save('results/ground_truth.npy', ps_true)
np.save('results/reconstruction.npy', recon)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print("\n" + "=" * 60)
npass = 'PASS' if metrics['numax_RE_pass'] else 'FAIL'
dpass = 'PASS' if metrics['delta_nu_RE_pass'] else 'FAIL'
print(f"νmax RE: {metrics['numax_relative_error']*100:.2f}% {npass}")
print(f"Δν RE:   {metrics['delta_nu_relative_error']*100:.2f}% {dpass}")
print(f"BG PSNR: {metrics['background_PSNR_dB']:.2f} dB")
print(f"Env CC:  {metrics['envelope_CC']:.4f}")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pysyd_astero**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=14.60 dB (power spectrum), SSIM=N/A (1D spectrum))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: pySYD: Asteroseismology
- Repository: https://github.com/ashleychontos/pySYD
